<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Structured Output
</b></font> </br></p>

---

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M05-Structured-Output"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

# 1 | Übersicht
---


Strukturierte Ausgaben sind keine Komfortfunktion — sie sind die Voraussetzung dafür, dass LLM-Ausgaben zuverlässig weiterverarbeitet werden können.

| Problem | Lösung |
|---|---|
| LLM gibt Freitext → schwer weiterverarbeitbar | `with_structured_output()` + Pydantic-Schema |
| Regex-Parsing → fehleranfällig | Automatische Validierung durch Pydantic |
| Kein Type-Checking | Typsichere Python-Objekte |


In [ ]:
#@markdown   <p><font size="4" color='green'>  Prozessdiagramm</font> </br></p>

diagram = '''
flowchart LR
    INPUT["Freier Text\n'Emma ist 34, Ärztin'"]
    SCHEMA["Pydantic Schema\nPerson(name, alter, beruf)"]
    LLM["LLM\ngpt-4o-mini"]
    OUTPUT["Python-Objekt\nPerson(name=Emma, alter=34, ...)"]

    INPUT --> LLM
    SCHEMA --> LLM
    LLM --> OUTPUT
'''

mermaid(diagram, width=750)

# 2 | Pydantic Basics
---



Pydantic ist die Grundlage für strukturierte Ausgaben. Ein **Pydantic-Modell** definiert:

- **Felder** mit Typ-Annotationen (`str`, `int`, `List[str]`, …)
- **Beschreibungen** per `Field(description=...)` – das LLM liest diese!
- **Optionale Felder** mit `Optional[...]` und `default=None`
- **Validierungslogik** automatisch durch Pydantic

> 💡 **Merkregel:**      
Der `description`-Text in `Field()` ist der Prompt-Hinweis für das LLM. Je klarer die Beschreibung, desto besser das Ergebnis.

In [ ]:
import json
from typing import Optional, List, Literal
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

# ── Konfigurationskonstanten ───────────────────────────────────────────────
MAX_RETRIES = 3   # API-Retry-Versuche bei kurzzeitige Störungen, die von selbst verschwinden (with_retry)

llm = init_chat_model(BASELINE, temperature=0.0)

# Einfaches Schema
class Person(BaseModel):
    """Informationen über eine Person."""
    name: str = Field(description="Vollständiger Name der Person")
    alter: int = Field(description="Alter der Person in Jahren")
    beruf: Optional[str] = Field(default=None, description="Aktueller Beruf, falls bekannt")

# Schema inspizieren – was das LLM sieht
mprint("## Pydantic-Schema")
mprint(f"**Felder:** `{list(Person.model_fields.keys())}`")

schema_json = json.dumps(Person.model_json_schema(), indent=2, ensure_ascii=False)
mprint(f"**JSON Schema:**\n```json\n{schema_json}\n```")
print(f"   MAX_RETRIES = {MAX_RETRIES}")

# 3 | with_structured_output()
---



`with_structured_output()` bindet ein Pydantic-Schema direkt ans LLM:

```python
structured_llm = llm.with_structured_output(MeinSchema)
ergebnis = structured_llm.invoke("...")
# ergebnis ist ein MeinSchema-Objekt – typsicher!
```

**Was intern passiert:**
1. LangChain schickt das JSON Schema des Pydantic-Modells an das LLM
2. Das LLM füllt die Felder anhand der `description`-Texte
3. LangChain validiert die Antwort und gibt ein typsicheres Objekt zurück

In [ ]:
class Person(BaseModel):
    """Informationen ueber eine Person."""
    name: str = Field(description="Vollstaendiger Name der Person")
    alter: int = Field(description="Alter der Person in Jahren")
    beruf: str = Field(description="Aktueller Beruf der Person")

# LLM mit Schema binden + with_retry gegen transiente API-Fehler
structured_llm = llm.with_structured_output(Person).with_retry(stop_after_attempt=MAX_RETRIES)
person_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m05_person_extraction_prompt.md", mode="T")

# Aufruf - LLM gibt ein Person-Objekt zurueck
ergebnis = (person_prompt | structured_llm).invoke(
    {"text": "Emma Mueller ist 34 Jahre alt und arbeitet als Ärztin."}
)

mprint("## Ergebnis")
mprint(f"**Typ:** `{type(ergebnis).__name__}` - kein String, sondern ein Python-Objekt")
mprint(f"**Name:** {ergebnis.name}")
mprint(f"**Alter:** {ergebnis.alter}")
mprint(f"**Beruf:** {ergebnis.beruf}")

# 4 | Praktische Anwendungen
---



Zwei häufige Anwendungsfälle:

**Informationsextraktion** – strukturierte Daten aus Freitext extrahieren

**Klassifikation** – Text in vordefinierte Kategorien einordnen (mit `Literal[...]`)

In [ ]:
class Ticket(BaseModel):
    """Klassifiziertes Support-Ticket."""
    kategorie: Literal["billing", "bug", "feature", "howto"] = Field(
        description="Kategorie: billing=Abrechnung, bug=Fehler, feature=Funktionswunsch, howto=Anleitung"
    )
    prioritaet: Literal["niedrig", "mittel", "hoch"] = Field(
        description="Prioritaet basierend auf Dringlichkeit und Auswirkung"
    )
    zusammenfassung: str = Field(
        description="Kurze Zusammenfassung des Anliegens in einem Satz"
    )

classifier = llm.with_structured_output(Ticket).with_retry(stop_after_attempt=MAX_RETRIES)
ticket_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m05_ticket_classification_prompt.md", mode="T")

tickets = [
    "Meine Kreditkarte wurde zweimal belastet!",
    "Die App stuerzt ab, wenn ich auf Speichern klicke.",
    "Waere es moeglich, einen Dark Mode hinzuzufuegen?",
]

mprint("## Klassifizierte Tickets")
for text in tickets:
    t = (ticket_prompt | classifier).invoke({"ticket": text})
    mprint("")
    mprint(f"**Ticket:** {text}")
    mprint(f"→ `{t.kategorie}` | Prioritaet: `{t.prioritaet}`")
    mprint(f"→ {t.zusammenfassung}")

# 5 | Structured Output in der Praxis
---



Structured Output eignet sich überall dort, wo das LLM-Ergebnis direkt weiterverarbeitet werden soll:
- Formular-Ausfüllung aus Freitext
- Analyse-Pipelines mit definierten Ausgabeformaten
- Bewertung und Scoring von Texten

Der Unterschied zu freiem Text: Das Ergebnis ist **sofort als Python-Objekt nutzbar** – kein Parsing, keine Fehlerbehandlung für Format-Abweichungen.

In [ ]:
class Adresse(BaseModel):
    """Postadresse."""
    strasse: str = Field(description="Strasse und Hausnummer")
    ort: str = Field(description="Ort oder Stadt")
    plz: Optional[str] = Field(default=None, description="Postleitzahl, falls genannt")

class Kontakt(BaseModel):
    """Extrahierte Kontaktdaten aus einem Text."""
    name: str = Field(description="Vollstaendiger Name")
    email: Optional[str] = Field(default=None, description="E-Mail-Adresse, falls genannt")
    telefon: Optional[str] = Field(default=None, description="Telefonnummer, falls genannt")
    adresse: Optional[Adresse] = Field(default=None, description="Adresse, falls genannt")

kontakt_llm = llm.with_structured_output(Kontakt).with_retry(stop_after_attempt=MAX_RETRIES)
kontakt_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m05_kontakt_extraction_prompt.md", mode="T")

text = """
Bitte kontaktieren Sie Dr. Anna Schmidt unter anna.schmidt@klinik.de
oder telefonisch unter 089 / 123 456 78.
Briefpost: Klinikstrasse 12, 80333 Muenchen.
"""

k = (kontakt_prompt | kontakt_llm).invoke({"text": text})

mprint("## Extrahierte Kontaktdaten")
mprint(f"**Name:** {k.name}")
mprint(f"**E-Mail:** {k.email}")
mprint(f"**Telefon:** {k.telefon}")
if k.adresse:
    mprint(f"**Adresse:** {k.adresse.strasse}, {k.adresse.plz} {k.adresse.ort}")

# 6 | LangSmith: Structured Output Traces
---



LangSmith macht Structured-Output-Aufrufe vollständig sichtbar:
- Das gesendete **JSON Schema** (Pydantic-Modell)
- Die **LLM-Antwort** vor der Validierung
- Das **valide Python-Objekt** nach der Validierung


In [ ]:
# 6.1 Structured Output Trace in LangSmith
class Produktbewertung(BaseModel):
    """Strukturierte Produktbewertung."""
    produkt: str = Field(description="Name des bewerteten Produkts")
    bewertung: int = Field(description="Bewertung von 1 (schlecht) bis 5 (sehr gut)")
    staerken: List[str] = Field(description="Positive Aspekte des Produkts (2-3 Punkte)")
    schwaechen: List[str] = Field(description="Negative Aspekte des Produkts (1-2 Punkte)")

# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M05_Kap6_StructuredTrace",
    "tags": ["m05", "structured-output", "langsmith"]
}

# 2. with_structured_output() + with_retry() + with_config()
bewertungs_llm = (
    llm
    .with_structured_output(Produktbewertung)
    .with_retry(stop_after_attempt=MAX_RETRIES)
    .with_config(**run_cfg)
)
bewertung_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m05_produktbewertung_prompt.md", mode="T")

rezension = """
Das MacBook Pro M3 ist ein beeindruckendes Geraet mit atemberaubender Performance
und hervorragender Akkulaufzeit. Der Preis ist allerdings sehr hoch und die
Anschlussauswahl bleibt nach wie vor limitiert.
"""
ergebnis = (bewertung_prompt | bewertungs_llm).invoke({"rezension": rezension})

mprint("## Trace erstellt")
mprint(f"**Produkt:** {ergebnis.produkt}")
mprint(f"**Bewertung:** {'⭐' * ergebnis.bewertung} ({ergebnis.bewertung}/5)")
mprint(f"**Staerken:** {', '.join(ergebnis.staerken)}")
mprint(f"**Schwaechen:** {', '.join(ergebnis.schwaechen)}")

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M05-Structured-Output", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, es kann aber auch gerne eine andere Herausforderung angegangen werden.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Wenn bei einer Aufgabe eine Blockade entsteht, kann zum Beispiel Gemini in Google Colab verwendet werden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

<p><font color='black' size="5">
Strukturierte Informationsextraktion für Ihren Arbeitskontext
</font></p>

Definieren Sie ein Pydantic-Schema fuer einen Dokument- oder Texttyp aus Ihrem Arbeitsumfeld (z. B. E-Mail, Bericht, Protokoll, Anfrage) und extrahieren Sie strukturierte Daten daraus.

**Grundlagen**
1. Definieren Sie ein Pydantic-Modell mit **mindestens 3 Feldern** (Thema frei waehlbar, z. B. Produkt, Bewerbung, Nachricht).
2. Erstellen Sie mit `llm.with_structured_output(IhrModell)` einen strukturierten LLM.
3. Extrahieren Sie Informationen aus einem Beispieltext und geben Sie das Ergebnis aus.

Speichern Sie das Modell in `mein_schema`, den strukturierten LLM in `mein_llm`.


**Aufbau**
1. Erweitern Sie das Schema auf **mindestens 4 Felder** – jedes Feld mit `Field(description=...)`.
2. Fuegen Sie **1 optionales Feld** (`Optional[...]`) hinzu.
3. Testen Sie den strukturierten LLM mit **3 unterschiedlichen Texten** (z. B. vollstaendig, lueckenhaft, mehrdeutig).
4. Pruefen Sie, wie das Modell mit fehlenden Informationen umgeht (optionales Feld `None`?).

Speichern Sie die 3 Test-Ergebnisse in `test_ergebnisse` (Liste von Pydantic-Objekten).


**Vertiefung**
1. Ergaenzen Sie ein `Literal[...]`-Feld fuer eine Klassifikation (z. B. `kategorie: Literal["positiv", "negativ", "neutral"]`).
2. Bauen Sie ein **Nested Schema**: Definieren Sie ein Sub-Modell und binden Sie es als Feld in das Haupt-Schema ein.
3. Kombinieren Sie Extraktion und Klassifikation in einem einzigen Schema und testen Sie es mit 2 Beispieltexten.
4. Aktivieren Sie LangSmith-Tracing, fuehren Sie einen Aufruf durch und inspizieren Sie im Trace, welches JSON-Schema an das LLM gesendet wird.

Speichern Sie das kombinierte Schema in `mein_kombinations_schema`.


<p><font color='black' size="5">
🔍 Selbstcheck mit `assert`
</font></p>

`assert` prüft eine Bedingung — ist sie `False`, stoppt Python mit einem `AssertionError` und zeigt die Fehlermeldung an:

```python
assert bedingung, "Fehlermeldung"

assert 2 + 2 == 4, "Rechnung falsch"    # ✅ kein Fehler
assert len("Hi") > 5, "Text zu kurz"   # ❌ AssertionError: Text zu kurz
```

**So nutzen Sie den Selbstcheck:**
1. Definieren Sie Ihr Pydantic-Schema und den strukturierten LLM in den Zellen über diesem Block
2. Speichern Sie die Pydantic-Klasse in **`mein_schema`** und den strukturierten LLM in **`mein_llm`**
3. Führen Sie die Zelle unten aus — Schema-Struktur und Test-Extraktion werden geprüft


<p><font color='black' size="5">
✅ Selbstcheck
</font></p>

In [ ]:
# ► Speichere dein Pydantic-Modell in 'mein_schema' und den strukturierten LLM in 'mein_llm'.

from pydantic import BaseModel

_schema = mein_schema  # ← Pydantic-Klasse anpassen
_llm    = mein_llm     # ← llm.with_structured_output(mein_schema) anpassen

assert issubclass(_schema, BaseModel), \
    "❌ mein_schema ist kein Pydantic BaseModel – prüfe 'class MeinSchema(BaseModel):'"

_felder = list(_schema.model_fields.keys())
assert len(_felder) >= 4, \
    f"❌ Nur {len(_felder)} Feld(er) – Mindestanforderung: ≥ 4 Felder"

_optional = [n for n, f in _schema.model_fields.items() if not f.is_required()]
assert len(_optional) >= 1, \
    "❌ Kein optionales Feld gefunden – mindestens 1 Feld mit Optional[...] oder default=... erforderlich"

_r = _llm.invoke("Max Mustermann, 42 Jahre, Berlin, Projekt-Manager, ab sofort verfügbar.")
assert isinstance(_r, _schema), \
    f"❌ Rückgabe ist kein {_schema.__name__} – prüfe with_structured_output(mein_schema)"

print(f"✅ Schema '{_schema.__name__}' · {len(_felder)} Felder ({len(_optional)} optional)")
print(f"   Felder: {_felder}")
print(f"   Test-Extraktion: {_r}")
